# Исследование рынка видеоигр в 2000–2013 годах

- Автор: Ефимов Алексей
- Дата: 11.05.26

### Цели и задачи проекта

<font color='#777778'>Цель проекта — подготовить данные о рынке видеоигр для дальнейшего исследования развития игровой индустрии в период с 2000 по 2013 год.

В рамках проекта необходимо изучить данные о видеоиграх, проверить их качество, обработать ошибки и пропуски, а также подготовить актуальный срез данных для анализа.

Основные задачи проекта:

1. Загрузить данные и познакомиться с их структурой.
2. Проверить названия столбцов и привести их к удобному стилю snake case.
3. Изучить типы данных и при необходимости преобразовать их.
4. Найти и обработать пропуски.
5. Проверить данные на наличие явных и неявных дубликатов.
6. Отобрать игры, выпущенные с 2000 по 2013 год включительно.
7. Категоризовать игры по оценкам пользователей и критиков.
8. Выделить топ-7 платформ по количеству выпущенных игр за актуальный период.
9. Сформулировать итоговый вывод о проделанной работе..</font>

### Описание данных

<font color='#777778'>В проекте используется датасет `/datasets/new_games.csv`, содержащий информацию о видеоиграх, выпущенных на разных платформах.

Описание столбцов:

- `Name` — название игры.
- `Platform` — название игровой платформы.
- `Year of Release` — год выпуска игры.
- `Genre` — жанр игры.
- `NA sales` — продажи в Северной Америке, млн копий.
- `EU sales` — продажи в Европе, млн копий.
- `JP sales` — продажи в Японии, млн копий.
- `Other sales` — продажи в других странах, млн копий.
- `Critic Score` — оценка критиков от 0 до 100.
- `User Score` — оценка пользователей от 0 до 10.
- `Rating` — возрастной рейтинг ESRB..</font>

### Содержимое проекта

<font color='#777778'>Проект состоит из следующих этапов:

1. Загрузка данных и знакомство с ними.
2. Проверка ошибок в данных и предобработка:
   - обработка названий столбцов;
   - проверка и изменение типов данных;
   - анализ и обработка пропусков;
   - поиск и обработка дубликатов.
3. Фильтрация данных за период с 2000 по 2013 год.
4. Категоризация данных:
   - по оценкам пользователей;
   - по оценкам критиков;
   - выделение топ-7 платформ.
5. Итоговый вывод..</font>

---

## 1. Загрузка данных и знакомство с ними



In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
# Путь на платформе Практикума или рядом с тетрадью при локальном запуске.
data_path = next(
    (path for path in (Path('/datasets/new_games.csv'), Path('new_games.csv'))
     if path.exists()),
    None
)
if data_path is None:
    raise FileNotFoundError('Поместите new_games.csv рядом с тетрадью.')

df = pd.read_csv(data_path)

In [3]:
df.head()

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


In [5]:
df.shape

(16956, 11)

In [6]:
df.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')

In [7]:
df.isna().sum()

Name                  2
Platform              0
Year of Release     275
Genre                 2
NA sales              0
EU sales              0
JP sales              0
Other sales           0
Critic Score       8714
User Score         6804
Rating             6871
dtype: int64

В датасете представлены данные о видеоиграх: название, платформа, год выпуска, жанр, продажи в разных регионах, оценки пользователей и критиков, а также возрастной рейтинг ESRB.

По результатам первичного знакомства видно, что данные в целом соответствуют описанию проекта. Однако уже на этом этапе можно отметить несколько особенностей:

- названия столбцов записаны не в едином стиле: есть заглавные буквы и пробелы;
- в некоторых столбцах есть пропущенные значения;
- некоторые числовые столбцы могут иметь некорректный тип данных;
- столбец с пользовательскими оценками может содержать строковые значения, поэтому его нужно дополнительно проверить;
- год выпуска игры может быть записан как вещественное число из-за пропусков.

Перед дальнейшей работой данные необходимо предобработать.

---

## 2.  Проверка ошибок в данных и их предобработка


### 2.1. Названия, или метки, столбцов датафрейма



In [8]:
df.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')

In [9]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r'\s+', '_', regex=True)
)

In [10]:
df.columns

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='object')

Названия столбцов были приведены к стилю snake case: теперь они записаны строчными буквами, а пробелы заменены на подчёркивания. Такой стиль удобнее использовать при обращении к столбцам в коде.

### 2.2. Типы данных


In [11]:
df.dtypes

name                object
platform            object
year_of_release    float64
genre               object
na_sales           float64
eu_sales            object
jp_sales            object
other_sales        float64
critic_score       float64
user_score          object
rating              object
dtype: object

In [12]:
df['user_score'].unique()

array(['8', nan, '8.3', '8.5', '6.6', '8.4', '8.6', '7.7', '6.3', '7.4',
       '8.2', '9', '7.9', '8.1', '8.7', '7.1', '3.4', '5.3', '4.8', '3.2',
       '8.9', '6.4', '7.8', '7.5', '2.6', '7.2', '9.2', '7', '7.3', '4.3',
       '7.6', '5.7', '5', '9.1', '6.5', 'tbd', '8.8', '6.9', '9.4', '6.8',
       '6.1', '6.7', '5.4', '4', '4.9', '4.5', '9.3', '6.2', '4.2', '6',
       '3.7', '4.1', '5.8', '5.6', '5.5', '4.4', '4.6', '5.9', '3.9',
       '3.1', '2.9', '5.2', '3.3', '4.7', '5.1', '3.5', '2.5', '1.9', '3',
       '2.7', '2.2', '2', '9.5', '2.1', '3.6', '2.8', '1.8', '3.8', '0',
       '1.6', '9.6', '2.4', '1.7', '1.1', '0.3', '1.5', '0.7', '1.2',
       '2.3', '0.5', '1.3', '0.2', '0.6', '1.4', '0.9', '1', '9.7'],
      dtype=object)

In [13]:
for column in ['eu_sales', 'jp_sales', 'user_score']:
    invalid = df[column].notna() & pd.to_numeric(df[column], errors='coerce').isna()
    print(f'{column}: {df.loc[invalid, column].value_counts().to_dict()}')

eu_sales: {'unknown': 6}
jp_sales: {'unknown': 4}
user_score: {'tbd': 2464}


In [14]:
numeric_columns = [
    'year_of_release', 'na_sales', 'eu_sales', 'jp_sales',
    'other_sales', 'critic_score', 'user_score'
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

In [15]:
df.dtypes

name                object
platform            object
year_of_release    float64
genre               object
na_sales           float64
eu_sales           float64
jp_sales           float64
other_sales        float64
critic_score       float64
user_score         float64
rating              object
dtype: object

В столбцах `eu_sales` и `jp_sales` встречается строка `unknown`, а в `user_score` — `tbd` (оценка ещё не определена). Из-за этого столбцы с числовыми данными имели тип `object`. Значения, которые нельзя преобразовать в числа, заменены на `NaN` с помощью `errors='coerce'`.

Год выпуска пока имеет вещественный тип из-за пропусков. После удаления строк без года он будет приведён к целочисленному типу.

### 2.3. Наличие пропусков в данных



In [16]:
missing_values = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': (df.isna().mean() * 100).round(2)
})

missing_values.sort_values(by='missing_count', ascending=False)

,missing_count,missing_percent
user_score,9268,54.66
critic_score,8714,51.39
rating,6871,40.52
year_of_release,275,1.62
eu_sales,6,0.04
jp_sales,4,0.02
name,2,0.01
genre,2,0.01
platform,0,0.00
na_sales,0,0.00


In [17]:
df[df['name'].isna()]

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
661,NaN,GEN,1993.0,NaN,1.78,0.53,0.00,0.08,NaN,NaN,NaN
14439,NaN,GEN,1993.0,NaN,0.00,0.00,0.03,0.00,NaN,NaN,NaN


In [18]:
initial_rows = len(df)
df = df.dropna(subset=['name', 'genre', 'year_of_release']).copy()
removed_for_missing = initial_rows - len(df)
df['year_of_release'] = df['year_of_release'].astype('int64')

# UNKNOWN отсутствует среди исходных возрастных рейтингов.
assert 'UNKNOWN' not in df['rating'].dropna().astype(str).str.upper().unique()
df['rating'] = df['rating'].fillna('UNKNOWN')

sales_columns = ['na_sales', 'eu_sales', 'jp_sales', 'other_sales']
for column in sales_columns:
    group_mean = df.groupby(['platform', 'year_of_release'])[column].transform('mean')
    df[column] = df[column].fillna(group_mean)
    df[column] = df[column].fillna(df[column].median())

df.isna().sum()

name                  0
platform              0
year_of_release       0
genre                 0
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       8594
user_score         9121
rating                0
dtype: int64

Пропуски обработаны следующим образом:

- строки с пропусками в `name`, `genre` или `year_of_release` удалены: без года нельзя определить, входит ли игра в нужный период;
- год выпуска после удаления этих строк преобразован к `int64`;
- пропуски в `rating` заменены на индикатор `UNKNOWN`, которого нет среди исходных обозначений ESRB;
- пропуски в региональных продажах заполнены средними по платформе и году выпуска, а при отсутствии среднего по группе — медианой столбца;
- пропуски в оценках пользователей и критиков оставлены как `NaN`, чтобы не искажать распределение оценок.

### 2.4. Явные и неявные дубликаты в данных

- Изучите уникальные значения в категориальных данных, например с названиями жанра игры, платформы, рейтинга и года выпуска. Проверьте, встречаются ли среди данных неявные дубликаты, связанные с опечатками или разным способом написания.
- При необходимости проведите нормализацию данных с текстовыми значениями. Названия или жанры игр можно привести к нижнему регистру, а названия рейтинга — к верхнему.

In [19]:
df['platform'].sort_values().unique()

array(['2600', '3DO', '3DS', 'DC', 'DS', 'GB', 'GBA', 'GC', 'GEN', 'GG',
       'N64', 'NES', 'NG', 'PC', 'PCFX', 'PS', 'PS2', 'PS3', 'PS4', 'PSP',
       'PSV', 'SAT', 'SCD', 'SNES', 'TG16', 'WS', 'Wii', 'WiiU', 'X360',
       'XB', 'XOne'], dtype=object)

In [20]:
df['genre'].sort_values().unique()

array(['ACTION', 'ADVENTURE', 'Action', 'Adventure', 'FIGHTING',
       'Fighting', 'MISC', 'Misc', 'PLATFORM', 'PUZZLE', 'Platform',
       'Puzzle', 'RACING', 'ROLE-PLAYING', 'Racing', 'Role-Playing',
       'SHOOTER', 'SIMULATION', 'SPORTS', 'STRATEGY', 'Shooter',
       'Simulation', 'Sports', 'Strategy'], dtype=object)

In [21]:
df['rating'].sort_values().unique()

array(['AO', 'E', 'E10+', 'EC', 'K-A', 'M', 'RP', 'T', 'UNKNOWN'],
      dtype=object)

In [22]:
df['name'] = df['name'].str.strip().str.lower()
df['genre'] = df['genre'].str.strip().str.lower()
df['platform'] = df['platform'].str.strip().str.upper()
df['rating'] = df['rating'].str.strip().str.upper()

In [23]:
df['platform'].sort_values().unique()

array(['2600', '3DO', '3DS', 'DC', 'DS', 'GB', 'GBA', 'GC', 'GEN', 'GG',
       'N64', 'NES', 'NG', 'PC', 'PCFX', 'PS', 'PS2', 'PS3', 'PS4', 'PSP',
       'PSV', 'SAT', 'SCD', 'SNES', 'TG16', 'WII', 'WIIU', 'WS', 'X360',
       'XB', 'XONE'], dtype=object)

In [24]:
df['genre'].sort_values().unique()

array(['action', 'adventure', 'fighting', 'misc', 'platform', 'puzzle',
       'racing', 'role-playing', 'shooter', 'simulation', 'sports',
       'strategy'], dtype=object)

In [25]:
df['rating'].sort_values().unique()

array(['AO', 'E', 'E10+', 'EC', 'K-A', 'M', 'RP', 'T', 'UNKNOWN'],
      dtype=object)

In [26]:
duplicates_count = df.duplicated().sum()
duplicates_count

np.int64(235)

In [27]:
df = df.drop_duplicates().reset_index(drop=True)

deleted_rows = initial_rows - len(df)
deleted_percent = deleted_rows / initial_rows * 100

print(f'Удалено строк без названия, жанра или года: {removed_for_missing}')
print(f'Удалено полных дубликатов: {duplicates_count}')
print(f'Всего удалено: {deleted_rows} ({deleted_percent:.2f}%)')
print(f'Осталось строк: {len(df)}')

Удалено строк без названия, жанра или года: 277
Удалено полных дубликатов: 235
Всего удалено: 512 (3.02%)
Осталось строк: 16444


Были изучены уникальные значения в категориальных столбцах `platform`, `genre` и `rating`.

Для устранения возможных неявных дубликатов текстовые значения были нормализованы:

- названия игр приведены к нижнему регистру;
- жанры приведены к нижнему регистру;
- платформы приведены к верхнему регистру;
- рейтинги ESRB приведены к верхнему регистру.

После этого были проверены явные дубликаты с помощью метода `duplicated()`. Найденные дубликаты были удалены методом `drop_duplicates()`.

Также было рассчитано количество и доля удалённых строк.

На этапе предобработки данные были подготовлены для дальнейшего анализа.

Были выполнены следующие действия:

- названия столбцов приведены к snake case;
- числовые столбцы с некорректными типами преобразованы с помощью `pd.to_numeric()`;
- строки с критичными пропусками удалены;
- пропуски в рейтинге ESRB заменены на значение-индикатор;
- пропуски в продажах обработаны с учётом платформы и года выпуска;
- текстовые значения нормализованы;
- проверены и удалены явные дубликаты;
- рассчитано количество удалённых строк.

Теперь данные можно фильтровать по нужному периоду и категоризовать.

---

## 3. Фильтрация данных

Коллеги хотят изучить историю продаж игр в начале XXI века, и их интересует период с 2000 по 2013 год включительно. Отберите данные по этому показателю. Сохраните новый срез данных в отдельном датафрейме, например `df_actual`.

In [28]:
df_actual = df[(df['year_of_release'] >= 2000) & (df['year_of_release'] <= 2013)].copy()

In [29]:
df_actual.shape

(12781, 11)

In [30]:
df_actual['year_of_release'].min(), df_actual['year_of_release'].max()

(np.int64(2000), np.int64(2013))

In [31]:
df_actual.head()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,wii sports,WII,2006,sports,41.36,28.96,3.77,8.45,76.0,8.0,E
2,mario kart wii,WII,2008,racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,wii sports resort,WII,2009,sports,15.61,10.93,3.28,2.95,80.0,8.0,E
6,new super mario bros.,DS,2006,platform,11.28,9.14,6.50,2.88,89.0,8.5,E
7,wii play,WII,2006,misc,13.96,9.18,2.93,2.84,58.0,6.6,E


Был создан новый датафрейм `df_actual`, в который вошли только игры, выпущенные с 2000 по 2013 год включительно.

Этот период соответствует задаче проекта: изучить развитие игровой индустрии в начале XXI века.

---

## 4. Категоризация данных
    


In [32]:
df_actual['user_score_category'] = pd.cut(
    df_actual['user_score'],
    bins=[0, 3, 8, 10],
    labels=['низкая оценка', 'средняя оценка', 'высокая оценка'],
    right=False,
    include_lowest=True
)

In [33]:
df_actual.loc[df_actual['user_score'] == 10, 'user_score_category'] = 'высокая оценка'

In [34]:
df_actual['user_score_category'] = df_actual['user_score_category'].cat.add_categories('нет оценки')
df_actual['user_score_category'] = df_actual['user_score_category'].fillna('нет оценки')

In [35]:
df_actual['user_score_category'].value_counts()

user_score_category
нет оценки        6298
средняя оценка    4081
высокая оценка    2286
низкая оценка      116
Name: count, dtype: int64

In [36]:
df_actual['critic_score_category'] = pd.cut(
    df_actual['critic_score'],
    bins=[0, 30, 80, 100],
    labels=['низкая оценка', 'средняя оценка', 'высокая оценка'],
    right=False,
    include_lowest=True
)

In [37]:
df_actual.loc[df_actual['critic_score'] == 100, 'critic_score_category'] = 'высокая оценка'

In [38]:
df_actual['critic_score_category'] = df_actual['critic_score_category'].cat.add_categories('нет оценки')
df_actual['critic_score_category'] = df_actual['critic_score_category'].fillna('нет оценки')

In [39]:
df_actual['critic_score_category'].value_counts()

critic_score_category
нет оценки        5612
средняя оценка    5422
высокая оценка    1692
низкая оценка       55
Name: count, dtype: int64

In [40]:
df_actual.groupby('user_score_category')['name'].count()

<cell 57>:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.


user_score_category
низкая оценка      116
средняя оценка    4081
высокая оценка    2286
нет оценки        6298
Name: name, dtype: int64

In [41]:
df_actual.groupby('critic_score_category')['name'].count()

critic_score_category
низкая оценка       55
средняя оценка    5422
высокая оценка    1692
нет оценки        5612
Name: name, dtype: int64

In [42]:
df_actual[['name', 'user_score', 'user_score_category', 'critic_score', 'critic_score_category']].head(10)

,name,user_score,user_score_category,critic_score,critic_score_category
0,wii sports,8.0,высокая оценка,76.0,средняя оценка
2,mario kart wii,8.3,высокая оценка,82.0,высокая оценка
3,wii sports resort,8.0,высокая оценка,80.0,высокая оценка
6,new super mario bros.,8.5,высокая оценка,89.0,высокая оценка
7,wii play,6.6,средняя оценка,58.0,средняя оценка
8,new super mario bros. wii,8.4,высокая оценка,87.0,высокая оценка
10,nintendogs,NaN,нет оценки,NaN,нет оценки
11,mario kart ds,8.6,высокая оценка,91.0,высокая оценка
13,wii fit,7.7,средняя оценка,80.0,высокая оценка
14,kinect adventures!,6.3,средняя оценка,61.0,средняя оценка


Игры были разделены на категории по пользовательским и экспертным оценкам.

Для пользовательских оценок:

- `низкая оценка` — от 0 до 3, не включая 3;
- `средняя оценка` — от 3 до 8, не включая 8;
- `высокая оценка` — от 8 до 10 включительно;
- `нет оценки` — если оценка отсутствует.

Для оценок критиков:

- `низкая оценка` — от 0 до 30, не включая 30;
- `средняя оценка` — от 30 до 80, не включая 80;
- `высокая оценка` — от 80 до 100 включительно;
- `нет оценки` — если оценка отсутствует.

Категоризация выполнена с помощью метода `pd.cut()`.

In [43]:
top_7_platforms = (
    df_actual['platform']
    .value_counts()
    .head(7)
)

top_7_platforms

platform
PS2     2127
DS      2120
WII     1275
PSP     1180
X360    1121
PS3     1087
GBA      811
Name: count, dtype: int64

In [44]:
top_7_platforms_list = top_7_platforms.index

In [45]:
df_actual['is_top_7_platform'] = df_actual['platform'].isin(top_7_platforms_list)

In [46]:
df_actual[['name', 'platform', 'is_top_7_platform']].head(10)

,name,platform,is_top_7_platform
0,wii sports,WII,True
2,mario kart wii,WII,True
3,wii sports resort,WII,True
6,new super mario bros.,DS,True
7,wii play,WII,True
8,new super mario bros. wii,WII,True
10,nintendogs,DS,True
11,mario kart ds,DS,True
13,wii fit,WII,True
14,kinect adventures!,X360,True


---

## 5. Итоговый вывод


В ходе проекта была выполнена предобработка данных о рынке видеоигр.

Сначала данные были загружены из файла `/datasets/new_games.csv`. Затем были изучены первые строки таблицы, общая информация о датафрейме, размер данных, названия столбцов и количество пропусков.

На этапе предобработки были выполнены следующие действия:

1. Названия столбцов приведены к стилю snake case.
2. Столбцы `user_score`, `critic_score` и `year_of_release` преобразованы к числовому типу.
3. Строки с пропусками в названии игры, жанре и годе выпуска удалены.
4. Пропуски в рейтинге ESRB заменены на значение `UNKNOWN`.
5. Пропуски в региональных продажах заполнены средними значениями по платформе и году выпуска.
6. Текстовые значения нормализованы для уменьшения риска неявных дубликатов.
7. Проверены и удалены явные дубликаты.
8. Посчитаны количество и доля удалённых строк.

После предобработки был создан датафрейм `df_actual`, содержащий игры, выпущенные с 2000 по 2013 год включительно. Этот срез соответствует цели проекта — изучить развитие игровой индустрии в начале XXI века.

Также были добавлены новые столбцы:

- `user_score_category` — категория пользовательской оценки;
- `critic_score_category` — категория оценки критиков;
- `is_top_7_platform` — признак того, входит ли платформа в топ-7 по количеству выпущенных игр.

Подготовленные данные можно использовать для дальнейшего анализа платформ, жанров, региональных продаж и особенностей RPG-игр.